For each NACE Class get the 100 chunks that scored highest across all the reports 

In [3]:
import pandas as pd
import glob
import os
import tqdm
import sys
import numpy as np
import matplotlib.pyplot as plt
sys.path.append("..")

#wor_dir = "/data/resources/weichel-llama3/work/projects/nace_classification/nace_report_topic_analysis"
#os.chdir(wor_dir)
wor_dir =" "
from test_base import *

In [4]:
overview_path = "/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/data/datasets/stoxx_600/stoxx_600_overview.csv"
overview_path = "/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/data/datasets/reports_subset_from_full_data_1/reports_subset_from_full_data_1_overview.csv"
overview_path = wor_dir + "/data/datasets/reports_subset_from_full_data_1/reports_subset_from_full_data_1_overview.csv"
overview_path = "/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/data/datasets/reports_subset_from_full_data_1/reports_subset_from_full_data_1_overview.csv"

In [5]:
raw_data_path = "/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/results/paragraph_and_sentence_len_3_min_chunk_len_100_cos_thresh_0.4_nace_level_2_stoxx/"
raw_data_path = "/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/results/paragraph_and_sentence_len_3_min_chunk_len_100_cos_thresh_0.4_nace_level_2_stoxx/"
raw_data_path = "/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/results/paragraph_and_sentence_len_3_min_chunk_len_0_cos_thresh_0.4_nace_level_1_stoxx/"
raw_data_path = "/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/results/tables_cos_sim_0.0_nace_level_1_stoxx/"
raw_data_path = "/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/results/sentence_len_5/paragraph_and_sentence_len_5_min_chunk_len_100_cos_thresh_0.4_nace_level_2_stoxx/"
raw_data_path = "/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/results/sentence_len_5/paragraph_and_sentence_len_5_min_chunk_len_100_cos_thresh_0.4_nace_level_1_stoxx/"
raw_data_path = "/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/results/paragraph_and_sentence_len_6_min_chunk_len_100_cos_thresh_0.4_nace_level_1_stoxx/"
raw_data_path = "/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/results/dataset__reports_subset_from_full_data_1_sentence_len_6__min_chunk_len_100__cos_thresh_0.4__nace_level_1/"
raw_data_path = "/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/results/dataset__reports_subset_from_full_data_1_sentence_len_6__min_chunk_len_100__cos_thresh_0.4__nace_level_2/"
raw_data_path = "/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/results/dataset__reports_subset_from_full_data_1_sentence_len_6__min_chunk_len_100__cos_thresh_0.4__nace_level_4/"

raw_data_path = "/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/results/dataset__reports_subset_from_full_data_1_sentence_len_6__min_chunk_len_100__cos_thresh_0.4__nace_level_1/"
raw_data_path = "/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/results/dataset__reports_subset_from_full_data_1_sentence_len_6__min_chunk_len_100__cos_thresh_0.4__nace_level_2/"
raw_data_path = wor_dir + "/results/dataset__reports_subset_from_full_data_1_sentence_len_6__min_chunk_len_100__cos_thresh_0.4__nace_level_4/"
raw_data_path = wor_dir + "/results/dataset__reports_subset_from_full_data_1_sentence_len_6__min_chunk_len_100__cos_thresh_0.4__nace_level_3/"
raw_data_path = "/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/results/dataset__reports_subset_from_full_data_1_sentence_len_6__min_chunk_len_100__cos_thresh_0.4__nace_level_2/"

In [6]:
reports = glob.glob(raw_data_path + "*/*_long.csv")
reports = glob.glob(raw_data_path + "*/*_short.csv")
len(reports)

1555

In [8]:
sample_ratio = 1

In [9]:
max_elements_per_class = 1000000

top_k_sentences = 200000

In [10]:
# if true, adds only chunks to training that have been classified into the same NACE class its report comes from
filter_only_right_chunks = True

In [11]:
# if true, adds random chunks that do not fulfill the minimum treshold (for BERT Training)
with_null_classifiers = True

In [12]:
new_threshold_cos_sin = 0.4

In [14]:
nace_level_descriptions = 2
nace_level = 1
assert nace_level_descriptions >= nace_level

In [15]:
training_data_path = "../data/training_data/"

In [16]:
suffix = f"sample_ratio_{sample_ratio}" + ("__filter_only_right_chunks" if filter_only_right_chunks else "") + ("__with_null_classifiers" if with_null_classifiers else "") + f"__nace_level_{nace_level}"

# "_subsample" if sample_ratio != 1 else ""
# end_path = os.path.join("/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/data/test_read_training_data_right_classifications", raw_data_path.split("/")[-2] + suffix)
# end_path = os.path.join("/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/data/test_read_training_data", raw_data_path.split("/")[-2] + suffix + "_with_no_class__cos_sim_04")
# end_path = os.path.join("/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/data/test_read_training_data_right_classifications", raw_data_path.split("/")[-2] + suffix + "_with_no_class__cos_sim_04")

end_path = os.path.join(training_data_path, 
                        raw_data_path.split("/")[-2] + "__" + suffix)
end_path

'../data/training_data/dataset__reports_subset_from_full_data_1_sentence_len_6__min_chunk_len_100__cos_thresh_0.4__nace_level_2__sample_ratio_1__filter_only_right_chunks__with_null_classifiers__nace_level_1'

In [17]:
#df_overview = pd.read_excel(overview_path)
df_overview = pd.read_csv(overview_path)
df_overview

,Unnamed: 0,Symbol,Name,Company is Active,Company Founded Date,Country of Primary Listing Iso3,CUSIP,Date Of First Trade,Entity Country HQ,Entity Credit Parent,...,ISIN,Primary Equity Listing,Proper Name,Public Company,Region Ticker,Sec is Primary Issue,Sec Type,SEDOL,NACE_letter,Report
0,5789,ZW0009011041,Ariston Holdings Ltd.,1,1947.0,ZWE,V97772103,20090317.0,ZWE,@NA,...,ZW0009011041,603408,Ariston Holdings Ltd.,1.0,ARIS-ZW,1,SHARE,6034081,A,Ariston Holdings Ltd.1.pdf
1,35816,INE978A01027,Heritage Foods Limited,1,1992.0,IND,Y3179H146,20020117.0,IND,06FQLY-E,...,INE978A01027,BF2F40,Heritage Foods Limited,1.0,519552-IN,1,SHARE,BF2F405,A,Heritage Foods Limited1.pdf
2,80373,MYL7854OO002,Timberwell Bhd.,1,1996.0,MYS,Y88399103,19970516.0,MYS,05JH15-E,...,MYL7854OO002,690556,Timberwell Bhd.,1.0,7854-MY,1,SHARE,6905563,A,Timberwell Bhd.1.pdf
3,49813,MYQ0189OO009,Matang Bhd.,1,2015.0,MYS,Y58347108,20170117.0,MYS,@NA,...,MYQ0189OO009,BYYQB5,Matang Bhd.,1.0,0189-MY,1,SHARE,BYYQB53,A,Matang Bhd.2.pdf
4,73064,MYL4316OO005,Sin Heng Chan (Malaya) Bhd.,1,1962.0,MYS,Y80178109,19880324.0,MYS,05YMQ5-E,...,MYL4316OO005,681088,Sin Heng Chan (Malaya) Bhd.,1.0,4316-MY,1,SHARE,6810883,A,Sin Heng Chan (Malaya) Bhd.1.pdf
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1557,5783,US0404432025,Aristocrat Group Corp.,0,2011.0,USA,040443202,20120313.0,USA,00DDP9-E,...,US0404432025,04044320,Aristocrat Group Corp.,1.0,ASCC-US,1,SHARE,BWX6257,S,Aristocrat Group Corp.1.pdf
1558,73228,KYG816BW1095,Sino-Life Group Limited,1,2005.0,HKG,G816BW109,20090909.0,HKG,00C60C-E,...,KYG816BW1095,B409GR,Sino-Life Group Limited,1.0,8296-HK,1,SHARE,B409GR3,S,Sino-Life Group Limited1.pdf
1559,86319,KYG9477E1070,Water Oasis Group Limited,1,1998.0,HKG,G9477E107,20070905.0,HKG,00610B-E,...,KYG9477E1070,651233,Water Oasis Group Limited,1.0,WOSSF-US,0,SHARE,B02V9Q9,S,Water Oasis Group Limited1.pdf
1560,66027,US76119X1054,"Reservoir Media, Inc.",1,2007.0,USA,76119X105,20210105.0,USA,0NVYZ2-E,...,US76119X1054,76119X10,"Reservoir Media, Inc.",1.0,RSVR-US,1,SHARE,BP0B9H7,S,"Reservoir Media, Inc.1.pdf"


In [18]:
df_nace_codes_descriptions = pd.read_csv("../data/NACE_Rev2_Structure_Explanatory_Notes_EN__1_.tsv", sep="\t")
filter_level_1_classes = "ABCDEFGHIJKLMNOPQRSTUVW"

In [19]:
result = pd.DataFrame(columns=["Sentences", "Score", "NACE_Code"])

# loop over all reports and get the all the sentences 
for report in tqdm.tqdm(reports):

    df = pd.read_csv(report)

    if filter_only_right_chunks: 
        report_name = os.path.basename(report).replace(".txt_short.csv", "") + ".pdf"
        report_code = df_overview[df_overview["Report"]==report_name]["NACE"].iloc[0]
        report_code = get_all_level(report_code)[nace_level]
        if nace_level == nace_level_descriptions: 
            filter_column = list(filter(lambda x: "Scores_"+str(report_code) in x, df.columns))
        else:
            filter_column = []
            for column in df.columns: 
                if "Scores" in column: 
                    if get_all_level(column.split("_")[1])[nace_level]==report_code: 
                        filter_column.append(column)
        df = df[["Sentences"]+filter_column]

    if nace_level == 1: 
        # filter some nace classes
        scores = df[[column for column in df.columns if ("Scores" in column) and get_all_level(column.split("_")[1], df_nace_codes_descriptions)[nace_level] in filter_level_1_classes]].columns
    else: 
        scores = df[[column for column in df.columns if ("Scores" in column)]].columns
    
    for score in scores:
        temp = df[df[score].notna()][["Sentences", score]]  
        try: 
            temp["NACE_Code"] = get_all_level(score.split("_")[1])[nace_level]
        except IndexError: 
            continue
        temp = temp.rename(columns={score: "Score"})
        result = pd.concat([result, temp])

  0%|          | 0/1555 [00:00<?, ?it/s]/var/folders/fp/yhl61lbj3m73x3_17tsp1vrr0000gn/T/ipykernel_30501/934534259.py:35: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  result = pd.concat([result, temp])
100%|██████████| 1555/1555 [00:56<00:00, 27.54it/s]


In [21]:
result

,Sentences,Score,NACE_Code
17,in june the company completed the sale of the ...,0.350486,C
25,for the thenowned arrow and geoffrey beene tra...,0.365972,C
26,of the costs incurred during million relates t...,0.330844,C
45,the companys management is responsible for mai...,0.342113,C
46,as a result we have received requests for sust...,0.329531,C
...,...,...,...
459,besides the adoption of alternative design for...,0.384183,F
460,he was a member of construction industry counc...,0.429248,F
470,depreciation allowance in of excess related de...,0.374058,F
477,plastic and chemical products building related...,0.435515,F


In [22]:
os.makedirs(end_path, exist_ok=True)

In [23]:
recordings = []

In [24]:
# for each code, store the 100 with the highest similarity score to the code

full_df = []
for code in set(result["NACE_Code"].to_list()): 

    #if not get_all_level(code.split("_")[1], df_nace_codes_descriptions)[1] in filter_level_1_classes: 
    # if not get_all_level(code, df_nace_codes_descriptions)[nace_level] in filter_level_1_classes: 
    #     continue

    temp = result[result["NACE_Code"] == code]
    temp = temp.drop_duplicates(subset="Sentences")
    temp = temp[temp["Sentences"].apply(len) >= 100]
    temp = temp.sort_values(by="Score", ascending=False)

    if with_null_classifiers: 
        temp.loc[temp["Score"]<new_threshold_cos_sin, "NACE_Code"] = "NO_CLASS"
        temp = temp[(temp["Score"] >= new_threshold_cos_sin) | (temp["NACE_Code"]=="NO_CLASS")]
        
        class_index = temp[temp["NACE_Code"]!="NO_CLASS"].index
        no_class_index = temp[temp["NACE_Code"]=="NO_CLASS"].index

        print(len(temp[temp["NACE_Code"]=="NO_CLASS"]))
        print(temp[temp["NACE_Code"]=="NO_CLASS"].index)
        print(temp.loc[no_class_index])
        print(code)
        print("--")

        temp = temp.loc[list(np.random.choice(no_class_index, len(class_index)))+list(class_index)]
    else:
        temp = temp[temp["Score"] >= new_threshold_cos_sin]

    number_of_elements_per_class = min(int(sample_ratio*len(temp)), max_elements_per_class, len(temp))
    random_choice = np.random.choice(len(temp), number_of_elements_per_class, replace=False)
    temp = temp.iloc[random_choice]
    temp = temp.sort_values(by="Score", ascending=False)
    temp = temp.iloc[:top_k_sentences, :]
    temp = temp.reset_index(drop=True)
    temp["Evaluation"] = None
    temp["Notes"] = None
    temp = temp[["Evaluation", "Notes", "Sentences", "Score", "NACE_Code"]]

    recordings.append({"Code": code,"Nbr. of Chunks": len(temp),"Avg. Length": temp["Sentences"].apply(len).mean(), "Avg. Score": temp["Score"].mean(), "Min. Score": temp["Score"].min(), "Max. Score": temp["Score"].max()})

    text = ""
    for i, row in temp.iterrows():
        text += f"#{i}, Score: " + str(round(row["Score"], 2)) + "\n\n" + row["Sentences"] + "\n\n"

    with open(os.path.join(end_path, code.replace("/"," ")) + ".txt", "w") as f:
        f.write(text)
    
    temp = temp.drop_duplicates(subset=["Sentences"])
    temp.to_csv(os.path.join(end_path, code.replace("/"," ")) + ".csv")

    full_df.append(temp)

full_df = pd.concat(full_df, axis=0, ignore_index=True)

9855
Index([111, 131, 135,  33, 147, 222, 300,  20, 318, 152,
       ...
        43,  34, 110,  96,  46,  29,   7,   6,  66,   9],
      dtype='int64', length=9855)
                                             Sentences     Score NACE_Code
111  glyphltcfontpophlatimesnewromanboldgteegglyphl...  0.356993  NO_CLASS
111  the campaign consisted of informational videos...  0.180872  NO_CLASS
111  during the period we recorded a yearonyear inc...  0.175545  NO_CLASS
111  shares included in other interests are benefic...  0.174614  NO_CLASS
111  on behalf of the board the board of directors ...  0.170880  NO_CLASS
..                                                 ...       ...       ...
9    the remaining aggregate principal with interes...  0.106504  NO_CLASS
9    revenue during the years ended december and re...  0.099678  NO_CLASS
9    significant transaction of boffice proposed du...  0.089605  NO_CLASS
9    company employee benefits and entitlements inc...  0.073958  NO_CLASS
9    wheth

In [35]:
statistics = full_df.groupby("NACE_Code").agg({"Sentences": "count", "Score": "mean"})

In [36]:
full_df = pd.concat([
    full_df[full_df["NACE_Code"] == "NO_CLASS"].sample(statistics.loc[statistics.index != "NO_CLASS", "Sentences"].max()), 
    full_df[full_df["NACE_Code"] != "NO_CLASS"]
    ])

In [37]:
statistics.to_csv(end_path + "/statistics.csv")

In [38]:
statistics

,Sentences,Score
NACE_Code,,
A,2273,0.449670
B,7582,0.458783
C,8274,0.441124
E,2318,0.470171
F,4789,0.457653
G,6166,0.437566
H,3358,0.446564
I,3964,0.447471
J,2243,0.439861


In [39]:
full_df= full_df.rename(columns={"Sentences": "text"})
#full_df = full_df.drop(columns="Score")
full_df

,Evaluation,Notes,text,Score,NACE_Code
114381,None,None,in moving towards sustainable supply chain pro...,0.284744,NO_CLASS
8016,None,None,fees equitysettled sharebased expense total re...,0.347439,NO_CLASS
31842,None,None,pages directors report sustainability report s...,0.319031,NO_CLASS
114336,None,None,the financial statements presented in this rep...,0.284953,NO_CLASS
14832,None,None,bsccl is an important member of two internatio...,0.317937,NO_CLASS
...,...,...,...,...,...
179811,None,None,the group has applied the practical expedient ...,0.413628,T
179812,None,None,material related party transactions entered in...,0.413411,T
179813,None,None,viii the entity or any member of a group of wh...,0.413084,T
179814,None,None,revenue is recognised when control over a prod...,0.408819,T


In [40]:
from sklearn.model_selection import train_test_split

# Split full_df into train (60%) and temp (40%)
train_df, temp_df = train_test_split(full_df, test_size=0.4, random_state=42)

# Split temp into test (20%) and validation (20%)
test_df, val_df = train_test_split(temp_df, test_size=0.5, random_state=42)

# Print the sizes of each split
print(f"Train size: {len(train_df)}, Test size: {len(test_df)}, Validation size: {len(val_df)}")

Train size: 47478, Test size: 15826, Validation size: 15826


In [41]:
full_df.to_csv(end_path + "/full_data.csv", index=False)

In [42]:
train_df.to_csv(end_path + "/train_data.csv", index=False)
val_df.to_csv(end_path + "/val_data.csv", index=False)
test_df.to_csv(end_path + "/test_data.csv", index=False)

In [43]:
end_path

'../data/training_data/dataset__reports_subset_from_full_data_1_sentence_len_6__min_chunk_len_100__cos_thresh_0.4__nace_level_2__sample_ratio_1__filter_only_right_chunks__with_null_classifiers__nace_level_1'

In [44]:
%pwd

'/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/benchmark'